In [1]:
print("hello world")

hello world


## Research Assistant Agent with Tools – First Step

#### Goal: Build an agent that can search the web (DuckDuckGo) and do math calculations. It decides which tool to use based on your question.

In [16]:
# !pip install duckduckgo-search

In [5]:
import langchain
import langchain_community
import langchain_openai
import duckduckgo_search

print(f"langchain version: {langchain.__version__}")
print(f"langchain_community version: {langchain_community.__version__}")
# print(f"langchain_openai version: {langchain_openai.__version__}")
print(f"duckduckgo_search version: {duckduckgo_search.__version__}")

langchain version: 1.2.17
langchain_community version: 0.4.1
duckduckgo_search version: 8.1.1


1. import required libraries

In [85]:
from langchain_openai import AzureChatOpenAI
from langchain.agents import create_agent   # type: ignore
from langchain_core.tools import tool   # type: ignore
from langchain_community.tools import DuckDuckGoSearchRun
# from langchain_community.tools import DuckDuckGoSearchResults
# from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage


1. define search tool

tool_1 - user define calculator for evaluation an math expression

In [78]:
@tool
def udf_calculator(expression: str) -> str:
    """use this tool to evaluate mathematical expressions. Input should be a math expression like '2+2'"""
    # evaluate an math expression and return the result as a string
    try:
        result = str(eval(expression))
    except Exception as err:
        result = f"Error evaluating expression: {err}"
    return result

tool_2 - web search to browse web to get required information

In [ ]:
web_search = DuckDuckGoSearchRun()
# web_search_result = DuckDuckGoSearchResults(output_format="list")  # type: ignore

enlists tools in tool set

In [80]:
# udf_tools = [
#     Tool(name="Web-Search", func=web_search.run, description="use this tool to search the web for current information"),
#     Tool(name="Calculator", func=udf_calculator, description="use this tool to evaluate mathematical expressions. Input should be a math expression like '2+2'")
# ]

udf_toolset = [web_search, udf_calculator]  # type: ignore





3. define the model

In [81]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

az_aoai_key = os.getenv("az_aoai_key")
az_model = os.getenv("az_model")
az_endpoint = os.getenv("az_endpoint")
api_ver = os.getenv("api_ver")



chat_model = AzureChatOpenAI(
    api_key=az_aoai_key,  # type: ignore
    api_version=api_ver,
    model=az_model,
    azure_endpoint=az_endpoint,
    temperature=0.01
)

4. define the system prompt

In [97]:
# sys_prompt = PromptTemplate.from_template(
#     """Answer the following question using the available tools.

#     Tools: {udf_tools}
#     Tool names = {tool_names}

#     You must follow this format
#     Question: the input question
#     Thought: reason about what to do
#     Action: the tool name to use (must be one of the {tool_names})
#     Action Input: the input to the tool
#     Observation: the tool's result
#     ... (repeat Thought/Action/Action Input/Observation as needed)
#     Thought: Now I know the final answer
#     Final Answer: the answer to the user question
    
#     Question: {input}
#     {agent_scratchpad}"""
# )


# from langchain_classic import hub
# sys_prompt = hub.pull("hwchase17/openai-tools-agent")    # type: ignore


# Update prompt to include agent_scratchpad
# sys_prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant. Use the following context to answer: {context}"),
#     ("placeholder", "{chat_history}"),
#     ("human", "{input}"),
#     ("placeholder", "{agent_scratchpad}"),
# ])

5. create agent to complete user task

In [93]:
# udf_agent = create_react_agent(chat_model,   # type: ignore 
#                                sys_prompt, 
#                                udf_tools)

agent_executor = create_agent(# type: ignore
    model=chat_model,
    tools=udf_toolset,
    checkpointer=mem,
    # verbose=True, 
    # max_iterations=3,
    # handle_parsing_errors=True
    )


# agent = create_agent(
#     model=chat_model,
#     tools=udf_toolset,
#     system_prompt="You are a helpful research assistant. Use tools to answer questions.",
#     verbose=True,
#     max_iterations=3,
#     handle_parsing_errors=True
# )



5B. integrate memory to system

In [94]:
from langgraph.checkpoint.memory import MemorySaver
mem = MemorySaver()

config = {"configurable": {"thread_id": "conversation-1"}}

6. execute the agent to response user query

In [99]:
while True:
    user_input = input("Enter your question (or 'exit' to quit): ")
    if user_input.lower() in ['exit', 'quit']:
        break
    # response = agent_executor.invoke({"user query": user_input})    # type: ignore
    response = agent_executor.stream({"messages":[HumanMessage(content=user_input)]}, stream_mode="values", config=config)    # type: ignore
    # response = agent.invoke({"messages":[HumanMessage(content=user_input)]})    # type: ignore
    for step in response:    # type: ignore
        # print(f"\n\tAgent step: {step}")
        # print(f"\n\tAgent response: {response['messages'][-1].content}")    # type: ignore
        print(f"\n\tAgent response: {step['messages'][-1]}")    # type: ignore

    


	Agent response: content='hello' additional_kwargs={} response_metadata={} id='74d1303e-c516-445e-b57d-26a85b0c95fd'

	Agent response: content='Hi there! How can I assist you today? 😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 976, 'total_tokens': 989, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 14, 'engine_ttft_ms': 149, 'engine_ttlt_ms': 336, 'pre_inference_ms': 70, 'service_tbt_ms': 15, 'service_ttft_ms': 507, 'service_ttlt_ms': 694, 'total_duration_ms': 631, 'user_visible_ttft_ms': 437}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-Dao1MEdW3pKAeh9HL5cYrTkvCKl0k', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'conte

## 🎯 Key Takeaways from Project 3

1. **`create_agent` is the modern replacement** for `create_react_agent` + `AgentExecutor` – simpler, one‑function API.

2. **`@tool` decorator** is cleaner than the old `Tool(name=..., func=...)` class.

3. **Message‑based input** (`{"messages": [("user", ...)]}`) is now the standard over plain dicts.

4. **Memory in agents** is added via `checkpointer=MemorySaver()` inside `create_agent` – no extra wrappers needed.

5. **Tool usage is visible** with `verbose=True` – you can see each `Action` and `Observation`.

6. **Agent decides tool usage autonomously** – you don't hardcode which tool to use; the LLM reasons based on the question.

7. **Prevent infinite loops** with `max_iterations` (pass directly to `create_agent`).

---

## ✅ What we can now build

- Research assistants with web search + calculations
- Internal tools (database queries, API calls) wrapped with `@tool`
- Conversational agents with long‑term memory (using `MemorySaver` or Redis)
